## 00.1 — Inspect raw archive

List files inside `data/raw/*.zip` and preview source headers. No data is changed.

In [2]:
from pathlib import Path
from zipfile import ZipFile

ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

raw_dir = ROOT / "data" / "raw"
zip_files = sorted(raw_dir.glob("*.zip"))

print("ZIP files found:", len(zip_files))

for zip_path in zip_files:
    print(f"\nArchive: {zip_path.name}")

    with ZipFile(zip_path) as z:
        members = [
            f for f in z.infolist()
            if not f.is_dir()
        ]

        print("Files inside:", len(members))

        for info in members[:30]:
            size_mb = info.file_size / (1024 ** 2)

            print(
                f"{info.filename} | "
                f"{size_mb:.2f} MB"
            )

            if info.filename.lower().endswith(".csv"):
                with z.open(info) as file:
                    header = file.readline().decode(
                        "utf-8-sig",
                        errors="replace"
                    )

                print("Header preview:", header[:400])

if not zip_files:
    print("\nPlace your raw ZIP inside data/raw/")

ZIP files found: 1

Archive: archive.zip
Files inside: 2
LCDataDictionary.xlsx | 0.02 MB
Loan_status_2007-2020Q3.gzip | 1691.31 MB


## 00.2 — Convert source to Parquet

Read the archived LendingClub data in chunks and save the unchanged source to `data/staging/`.

In [3]:
from pathlib import Path
from zipfile import ZipFile

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

ZIP_PATH = ROOT / "data/raw/archive.zip"
CSV_MEMBER = "Loan_status_2007-2020Q3.gzip"

OUT_DIR = ROOT / "data/staging"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = OUT_DIR / "lendingclub_full_raw.parquet"
TEMP_PATH = OUT_DIR / "lendingclub_full_raw.partial"

CHUNK_SIZE = 50_000

writer = None
total_rows = 0

try:
    with ZipFile(ZIP_PATH) as archive:
        with archive.open(CSV_MEMBER) as source:

            reader = pd.read_csv(
                source,
                chunksize=CHUNK_SIZE,
                dtype=str,
                na_filter=False,
                encoding="utf-8-sig"
            )

            for chunk_number, chunk in enumerate(reader, start=1):
                table = pa.Table.from_pandas(
                    chunk,
                    preserve_index=False
                )

                if writer is None:
                    writer = pq.ParquetWriter(
                        TEMP_PATH,
                        table.schema,
                        compression="zstd"
                    )

                writer.write_table(table)
                total_rows += len(chunk)

                if chunk_number % 5 == 0:
                    print(f"Processed: {total_rows:,} rows")

    if writer is None:
        raise ValueError("No data rows found.")

    writer.close()
    writer = None

    TEMP_PATH.replace(OUT_PATH)

except Exception:
    if writer is not None:
        writer.close()
    raise

# Verify the saved file
parquet = pq.ParquetFile(OUT_PATH)

print("\nConversion completed")
print("Rows:", f"{parquet.metadata.num_rows:,}")
print("Columns:", parquet.metadata.num_columns)
print("Size (GB):", round(OUT_PATH.stat().st_size / 1024**3, 2))
print("Saved to:", OUT_PATH)

assert parquet.metadata.num_rows == total_rows

Processed: 250,000 rows
Processed: 500,000 rows
Processed: 750,000 rows
Processed: 1,000,000 rows
Processed: 1,250,000 rows
Processed: 1,500,000 rows
Processed: 1,750,000 rows
Processed: 2,000,000 rows
Processed: 2,250,000 rows
Processed: 2,500,000 rows
Processed: 2,750,000 rows

Conversion completed
Rows: 2,925,493
Columns: 142
Size (GB): 0.37
Saved to: d:\ML_Projects\P2P-Lending-Default-Prediction\data\staging\lendingclub_full_raw.parquet


## 00.3 — Validate source and dictionary

Confirm Parquet dimensions, inspect the supplied data dictionary, and preview raw columns.

In [4]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO

import pandas as pd
import polars as pl
import pyarrow.parquet as pq
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_PARQUET = ROOT / "data/staging/lendingclub_full_raw.parquet"
ZIP_PATH = ROOT / "data/raw/archive.zip"

assert RAW_PARQUET.exists(), "Full-source Parquet not found."

# 1. Verify Parquet metadata (does not load entire dataset)
metadata = pq.ParquetFile(RAW_PARQUET).metadata

print("FULL SOURCE")
print("Rows:", f"{metadata.num_rows:,}")
print("Columns:", metadata.num_columns)
print("Size (GB):", round(RAW_PARQUET.stat().st_size / 1024**3, 2))

# 2. Read dictionary directly from ZIP
with ZipFile(ZIP_PATH) as archive:
    dictionary_bytes = archive.read("LCDataDictionary.xlsx")

dictionary_sheets = pd.read_excel(
    BytesIO(dictionary_bytes),
    sheet_name=None
)

print("\nDATA DICTIONARY")

for sheet_name, sheet in dictionary_sheets.items():
    print(f"\nSheet: {sheet_name}")
    print("Dimensions:", sheet.shape)
    display(sheet.head(10))

# 3. Lazy-load full source
lf = pl.scan_parquet(RAW_PARQUET)

print("\nSCHEMA")
print("Total columns:", len(lf.collect_schema().names()))

print("\nSAMPLE RECORDS")
display(
    lf.select([
        "id",
        "loan_amnt",
        "term",
        "annual_inc",
        "issue_d",
        "loan_status",
        "fico_range_low",
        "dti",
    ])
    .head(5)
    .collect()
    .to_pandas()
)

FULL SOURCE
Rows: 2,925,493
Columns: 142
Size (GB): 0.37

DATA DICTIONARY

Sheet: LoanStats
Dimensions: (153, 2)


,LoanStatNew,Description
0,acc_now_delinq,The number of accounts on which the borrower i...
1,acc_open_past_24mths,Number of trades opened in past 24 months.
2,addr_state,The state provided by the borrower in the loan...
3,all_util,Balance to credit limit on all trades
4,annual_inc,The self-reported annual income provided by th...
5,annual_inc_joint,The combined self-reported annual income provi...
6,application_type,Indicates whether the loan is an individual ap...
7,avg_cur_bal,Average current balance of all accounts
8,bc_open_to_buy,Total open to buy on revolving bankcards.
9,bc_util,Ratio of total current balance to high credit/...



SCHEMA
Total columns: 142

SAMPLE RECORDS


,id,loan_amnt,term,annual_inc,issue_d,loan_status,fico_range_low,dti
0,1077501,5000,36 months,24000,Dec-2011,Fully Paid,735,27.65
1,1077430,2500,60 months,30000,Dec-2011,Charged Off,740,1
2,1077175,2400,36 months,12252,Dec-2011,Fully Paid,735,8.72
3,1076863,10000,36 months,49200,Dec-2011,Fully Paid,690,20
4,1075358,3000,60 months,80000,Dec-2011,Fully Paid,695,17.94


## 00.4 — Audit status, term, and dates

Inspect loan-status and term distributions, date formats, and essential-field blanks. Export two CSV summaries.

In [5]:

# 1. Loan status distribution
status_counts = (
    lf.group_by("loan_status")
    .agg(pl.len().alias("loans"))
    .sort("loans", descending=True)
    .collect()
)

print("LOAN STATUS DISTRIBUTION")
display(status_counts.to_pandas())


# 2. Loan term distribution
term_counts = (
    lf.group_by("term")
    .agg(pl.len().alias("loans"))
    .sort("loans", descending=True)
    .collect()
)

print("\nLOAN TERM DISTRIBUTION")
display(term_counts.to_pandas())


# 3. Inspect original issue-date formatting
date_samples = (
    lf.select("issue_d")
    .unique()
    .sort("issue_d")
    .collect()
)

print("\nISSUE DATE EXAMPLES")
display(date_samples.head(12).to_pandas())
display(date_samples.tail(12).to_pandas())


# 4. Check blank values in essential cohort columns
required_columns = [
    "id", "issue_d", "term", "loan_status"
]

checks = [
    (
        pl.col(c).is_null()
        | (pl.col(c).str.strip_chars() == "")
    )
    .fill_null(True)
    .sum()
    .alias(c + "_blank")
    for c in required_columns
]

print("\nESSENTIAL FIELD CHECKS")
essential_counts_df = lf.select(checks).collect().to_pandas()
display(essential_counts_df)


# CSV REPORTS: source distributions and essential-field quality
AUDIT_REPORT_DIR = ROOT / "reports" / "data_audit"
AUDIT_REPORT_DIR.mkdir(parents=True, exist_ok=True)

source_rows = int(status_counts["loans"].sum())

status_report = status_counts.to_pandas().rename(
    columns={"loan_status": "value"}
)
status_report.insert(0, "dimension", "loan_status")

term_report = term_counts.to_pandas().rename(
    columns={"term": "value"}
)
term_report.insert(0, "dimension", "term")

source_report = pd.concat(
    [status_report, term_report], ignore_index=True
)

source_report["percentage"] = (
    100 * source_report["loans"] / source_rows
).round(2)

source_report = source_report[
    ["dimension", "value", "loans", "percentage"]
]

source_report_path = AUDIT_REPORT_DIR / "03_source_distributions.csv"
source_report.to_csv(source_report_path, index=False)
print("Saved audit report:", source_report_path)


essential_quality_report = pd.DataFrame([
    {
        "column": col,
        "total_rows": source_rows,
        "blank_rows": int(essential_counts_df.iloc[0][col + "_blank"]),
        "nonblank_rows": source_rows - int(
            essential_counts_df.iloc[0][col + "_blank"]
        ),
        "blank_percentage": round(
            100 * int(essential_counts_df.iloc[0][col + "_blank"])
            / source_rows, 2
        ),
    }
    for col in required_columns
])

quality_report_path = AUDIT_REPORT_DIR / "04_essential_field_quality.csv"
essential_quality_report.to_csv(quality_report_path, index=False)
print("Saved audit report:", quality_report_path)


LOAN STATUS DISTRIBUTION


,loan_status,loans
0,Fully Paid,1497783
1,Current,1031016
2,Charged Off,362548
3,Late (31-120 days),16154
4,In Grace Period,10028
5,Late (16-30 days),2719
6,Issued,2062
7,Does not meet the credit policy. Status:Fully ...,1988
8,Does not meet the credit policy. Status:Charge...,761
9,Default,433



LOAN TERM DISTRIBUTION


,term,loans
0,36 months,2060077
1,60 months,865415
2,,1



ISSUE DATE EXAMPLES


,issue_d
0,
1,Apr-2008
2,Apr-2009
3,Apr-2010
4,Apr-2011
5,Apr-2012
6,Apr-2013
7,Apr-2014
8,Apr-2015
9,Apr-2016


,issue_d
0,Sep-2009
1,Sep-2010
2,Sep-2011
3,Sep-2012
4,Sep-2013
5,Sep-2014
6,Sep-2015
7,Sep-2016
8,Sep-2017
9,Sep-2018



ESSENTIAL FIELD CHECKS


,id_blank,issue_d_blank,term_blank,loan_status_blank
0,0,1,1,1


Saved audit report: d:\ML_Projects\P2P-Lending-Default-Prediction\reports\data_audit\03_source_distributions.csv
Saved audit report: d:\ML_Projects\P2P-Lending-Default-Prediction\reports\data_audit\04_essential_field_quality.csv


## 00.5 — Assess candidate cohort

Compare year-wise loan resolution and nine feature-availability rates for 36-month loans. Export two CSV summaries.

In [6]:
import polars as pl
import pandas as pd
from IPython.display import display

lf = pl.scan_parquet(
    ROOT / "data/staging/lendingclub_full_raw.parquet"
)

# Parse date and term without changing source data
date_text = pl.col("issue_d").str.strip_chars()

base = (
    lf.with_columns(
        pl.coalesce(
            date_text.str.strptime(
                pl.Date, "%Y-%m-%d", strict=False
            ),
            pl.concat_str([date_text, pl.lit("-01")])
              .str.strptime(pl.Date, "%b-%Y-%d", strict=False),
            pl.concat_str([date_text, pl.lit("-01")])
              .str.strptime(pl.Date, "%b-%y-%d", strict=False),
        ).alias("_issue_date"),

        pl.col("term")
          .str.extract(r"(\d+)", 1)
          .cast(pl.Int16, strict=False)
          .alias("_term_months"),

        pl.col("loan_status")
          .str.strip_chars()
          .alias("_status")
    )
    .with_columns(
        pl.col("_issue_date").dt.year().alias("_year")
    )
)

loans_36m = base.filter(
    (pl.col("_term_months") == 36)
    & pl.col("_year").is_between(2011, 2020)
)

# A. Loan counts, outcome availability and default rates
year_summary = (
    loans_36m
    .group_by("_year")
    .agg(
        pl.len().alias("total"),
        (pl.col("_status") == "Fully Paid")
          .sum().alias("fully_paid"),
        (pl.col("_status") == "Charged Off")
          .sum().alias("charged_off"),
    )
    .with_columns(
        (
            pl.col("fully_paid") + pl.col("charged_off")
        ).alias("resolved")
    )
    .with_columns(
        (pl.col("total") - pl.col("resolved"))
          .alias("other_statuses"),

        (100 * pl.col("resolved") / pl.col("total"))
          .round(2).alias("resolved_pct"),

        (100 * pl.col("charged_off") / pl.col("resolved"))
          .round(2).alias("default_pct_resolved")
    )
    .sort("_year")
    .collect()
)

print("A. 36-MONTH LOANS BY ISSUE YEAR")
display(year_summary.to_pandas())


# B. Availability of representative credit-report features
features = [
    "annual_inc",
    "dti",
    "fico_range_low",
    "bc_util",
    "tot_cur_bal",
    "mort_acc",
    "pct_tl_nvr_dlq",
    "open_acc_6m",
    "inq_last_12m",
]

features = [
    c for c in features
    if c in lf.collect_schema()
]

coverage_expr = []

for c in features:
    s = pl.col(c).str.strip_chars().str.to_lowercase()

    valid = (
        s.is_not_null()
        & ~s.is_in(["", "-1", "nan", "null", "none", "n/a"])
    ).fill_null(False)

    coverage_expr.append(
        (100 * valid.mean()).round(2).alias(c)
    )

availability = (
    loans_36m
    .filter(pl.col("_year").is_between(2012, 2019))
    .group_by("_year")
    .agg(coverage_expr)
    .sort("_year")
    .collect()
)

print("\nB. FEATURE AVAILABILITY (%) BY YEAR")
display(
    availability.to_pandas()
    .set_index("_year")
    .T
)


# CSV REPORTS: year-wise loan resolution and feature availability
AUDIT_REPORT_DIR = ROOT / "reports" / "data_audit"
AUDIT_REPORT_DIR.mkdir(parents=True, exist_ok=True)

yearly_resolution_report = year_summary.to_pandas().rename(
    columns={"_year": "issue_year"}
)
resolution_report_path = AUDIT_REPORT_DIR / "05_yearly_loan_resolution.csv"
yearly_resolution_report.to_csv(resolution_report_path, index=False)
print("Saved audit report:", resolution_report_path)

feature_availability_report = (
    availability.to_pandas()
    .melt(
        id_vars="_year",
        var_name="feature",
        value_name="availability_pct",
    )
    .rename(columns={"_year": "issue_year"})
    .sort_values(["issue_year", "feature"])
    .reset_index(drop=True)
)
availability_report_path = AUDIT_REPORT_DIR / "06_feature_availability.csv"
feature_availability_report.to_csv(availability_report_path, index=False)
print("Saved audit report:", availability_report_path)


A. 36-MONTH LOANS BY ISSUE YEAR


,_year,total,fully_paid,charged_off,resolved,other_statuses,resolved_pct,default_pct_resolved
0,2011,14101,12602,1499,14101,0,100.00,10.63
1,2012,43470,37567,5903,43470,0,100.00,13.58
2,2013,100422,88044,12378,100422,0,100.00,12.33
3,2014,162570,140255,22315,162570,0,100.00,13.73
4,2015,283173,240995,42178,283173,0,100.00,14.89
5,2016,323495,272076,51299,323375,120,99.96,15.86
6,2017,320419,204518,42563,247081,73338,77.11,17.23
7,2018,344671,116488,28919,145407,199264,42.19,19.89
8,2019,349414,46576,8875,55451,293963,15.87,16.01
9,2020,100909,3219,35,3254,97655,3.22,1.08



B. FEATURE AVAILABILITY (%) BY YEAR


_year,2012,2013,2014,2015,2016,2017,2018,2019
annual_inc,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
dti,100.00,100.00,100.00,100.00,99.99,99.90,99.81,99.83
fico_range_low,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
bc_util,85.11,99.16,98.84,98.93,98.79,98.57,98.56,98.65
tot_cur_bal,48.13,100.00,100.00,100.00,100.00,100.00,100.00,100.00
mort_acc,86.14,100.00,100.00,100.00,100.00,100.00,100.00,100.00
pct_tl_nvr_dlq,48.13,99.89,100.00,100.00,100.00,100.00,100.00,100.00
open_acc_6m,0.00,0.00,0.00,5.27,99.99,100.00,100.00,100.00
inq_last_12m,0.00,0.00,0.00,5.27,99.99,100.00,100.00,100.00


Saved audit report: d:\ML_Projects\P2P-Lending-Default-Prediction\reports\data_audit\05_yearly_loan_resolution.csv
Saved audit report: d:\ML_Projects\P2P-Lending-Default-Prediction\reports\data_audit\06_feature_availability.csv


## 00.6 — Create resolved-loan cohort

Keep 2013–2016, 36-month Fully Paid (0) and Charged Off (1) loans. Save the raw cohort Parquet without imputing or scaling; resolved-only selection may bias outcomes.

In [7]:
import polars as pl
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

SOURCE = ROOT / "data/staging/lendingclub_full_raw.parquet"

OUT_DIR = ROOT / "data/interim"
OUT_DIR.mkdir(parents=True, exist_ok=True)

COHORT_PATH = OUT_DIR / "lendingclub_2013_2016_36m_raw.parquet"
TEMP_PATH = OUT_DIR / "lendingclub_2013_2016_36m_raw.partial.parquet"

lf = pl.scan_parquet(SOURCE)

# Source fields are strings at this stage
raw_date = pl.col("issue_d").str.strip_chars()

# Support common LendingClub date formats
issue_date = pl.coalesce(
    raw_date.str.strptime(
        pl.Date, "%Y-%m-%d", strict=False
    ),
    pl.concat_str([raw_date, pl.lit("-01")]).str.strptime(
        pl.Date, "%b-%Y-%d", strict=False
    ),
    pl.concat_str([raw_date, pl.lit("-01")]).str.strptime(
        pl.Date, "%b-%y-%d", strict=False
    ),
    pl.concat_str([raw_date, pl.lit("-01")]).str.strptime(
        pl.Date, "%Y-%m-%d", strict=False
    ),
)

base = lf.with_columns(
    issue_date.alias("issue_date"),
    pl.col("term")
      .str.extract(r"(\d+)", 1)
      .cast(pl.Int16, strict=False)
      .alias("term_months"),
    pl.col("loan_status")
      .str.strip_chars()
      .alias("_normalized_status"),
)

base = base.with_columns(
    pl.col("issue_date").dt.year().alias("issue_year")
)

# Apply cohort criteria
cohort = (
    base
    .filter(
        pl.col("issue_year").is_between(2013, 2016)
        & (pl.col("term_months") == 36)
        & pl.col("_normalized_status").is_in(
            ["Fully Paid", "Charged Off"]
        )
    )
    .with_columns(
        pl.when(pl.col("_normalized_status") == "Charged Off")
          .then(1)
          .otherwise(0)
          .cast(pl.Int8)
          .alias("target")
    )
    .drop("_normalized_status")
)

# Write without loading the entire cohort into memory
cohort.sink_parquet(
    TEMP_PATH,
    compression="zstd"
)

TEMP_PATH.replace(COHORT_PATH)

print("Cohort exported successfully")
print("Saved to:", COHORT_PATH)

Cohort exported successfully
Saved to: d:\ML_Projects\P2P-Lending-Default-Prediction\data\interim\lendingclub_2013_2016_36m_raw.parquet


## 00.7 — Validate cohort

Check row counts, target balance, issue years, terms, and null targets. Export target and validation CSV summaries.

In [8]:

import pyarrow.parquet as pq
from IPython.display import display

df = pl.scan_parquet(COHORT_PATH)

# 1. Verify dimensions
metadata = pq.ParquetFile(COHORT_PATH).metadata

print("COHORT DIMENSIONS")
print("Rows:", f"{metadata.num_rows:,}")
print("Columns:", metadata.num_columns)

# 2. Target distribution
target_summary = (
    df.group_by("target")
    .agg(pl.len().alias("loans"))
    .with_columns(
        (100 * pl.col("loans") / pl.col("loans").sum())
        .round(2)
        .alias("percentage")
    )
    .sort("target")
    .collect()
)

print("\nTARGET DISTRIBUTION")
display(target_summary.to_pandas())

# 3. Year-wise cohort distribution
year_summary = (
    df.group_by(["issue_year", "loan_status"])
    .agg(pl.len().alias("loans"))
    .sort(["issue_year", "loan_status"])
    .collect()
)

print("\nYEAR-WISE DISTRIBUTION")
display(year_summary.to_pandas())

# 4. Validate cohort conditions
checks = df.select(
    pl.len().alias("total_rows"),
    pl.col("issue_year").is_null().sum().alias("invalid_dates"),
    (pl.col("term_months") != 36).sum().alias("wrong_term"),
    (~pl.col("issue_year").is_between(2013, 2016))
      .sum().alias("wrong_year"),
    pl.col("target").is_null().sum().alias("missing_targets"),
).collect()

print("\nVALIDATION")
display(checks.to_pandas())


# CSV REPORTS: target breakdown and cohort validation
import pandas as pd

AUDIT_REPORT_DIR = ROOT / "reports" / "data_audit"
AUDIT_REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Overall class proportions from the existing target summary
overall_target_report = target_summary.to_pandas()
overall_target_report["scope"] = "overall"
overall_target_report["issue_year"] = ""
overall_target_report["loan_status"] = overall_target_report["target"].map({
    0: "Fully Paid", 1: "Charged Off"
})

# Per-year percentages computed from the already-calculated year summary
yearly_target_report = year_summary.to_pandas()
yearly_target_report["scope"] = "by_issue_year"
yearly_target_report["target"] = yearly_target_report["loan_status"].map({
    "Fully Paid": 0, "Charged Off": 1
})

yearly_target_report["percentage"] = (
    100 * yearly_target_report["loans"]
    / yearly_target_report.groupby("issue_year")["loans"].transform("sum")
).round(2)

yearly_target_report["issue_year"] = (
    yearly_target_report["issue_year"].astype(str)
)

cohort_target_report = pd.concat(
    [overall_target_report, yearly_target_report], ignore_index=True
)[["scope", "issue_year", "target", "loan_status", "loans", "percentage"]]

target_report_path = AUDIT_REPORT_DIR / "07_cohort_target_distribution.csv"
cohort_target_report.to_csv(target_report_path, index=False)
print("Saved audit report:", target_report_path)


# Preserve all four existing validation counts and their PASS/FAIL status
check_values = checks.to_pandas().iloc[0]

validation_report = pd.DataFrame([
    {
        "check": check_name,
        "total_rows": int(check_values["total_rows"]),
        "affected_rows": int(check_values[check_name]),
        "status": (
            "PASS" if int(check_values[check_name]) == 0 else "FAIL"
        ),
    }
    for check_name in [
        "invalid_dates", "wrong_term", "wrong_year", "missing_targets"
    ]
])

validation_report_path = AUDIT_REPORT_DIR / "08_cohort_validation.csv"
validation_report.to_csv(validation_report_path, index=False)
print("Saved audit report:", validation_report_path)


assert checks["invalid_dates"][0] == 0
assert checks["wrong_term"][0] == 0
assert checks["wrong_year"][0] == 0
assert checks["missing_targets"][0] == 0

assert set(
    target_summary["target"].to_list()
) == {0, 1}

print("\nPASS: Cohort conditions are satisfied.")


COHORT DIMENSIONS
Rows: 869,540
Columns: 146

TARGET DISTRIBUTION


,target,loans,percentage
0,0,741370,85.26
1,1,128170,14.74



YEAR-WISE DISTRIBUTION


,issue_year,loan_status,loans
0,2013,Charged Off,12378
1,2013,Fully Paid,88044
2,2014,Charged Off,22315
3,2014,Fully Paid,140255
4,2015,Charged Off,42178
5,2015,Fully Paid,240995
6,2016,Charged Off,51299
7,2016,Fully Paid,272076



VALIDATION


,total_rows,invalid_dates,wrong_term,wrong_year,missing_targets
0,869540,0,0,0,0


Saved audit report: d:\ML_Projects\P2P-Lending-Default-Prediction\reports\data_audit\07_cohort_target_distribution.csv
Saved audit report: d:\ML_Projects\P2P-Lending-Default-Prediction\reports\data_audit\08_cohort_validation.csv

PASS: Cohort conditions are satisfied.


## 00.8 — Full-column missingness

Audit every source and cohort column for nulls, blanks, and textual missing tokens. Report `-1` separately because its meaning varies by feature; export one CSV.

In [9]:
# 09 — Full-column missingness audit (source + selected cohort)
# This report does not modify either dataset.
import pandas as pd
import polars as pl
import pyarrow.parquet as pq
from IPython.display import display

AUDIT_REPORT_DIR = ROOT / "reports" / "data_audit"
AUDIT_REPORT_DIR.mkdir(parents=True, exist_ok=True)

missingness_datasets = {
    "full_source": RAW_PARQUET,
    "resolved_2013_2016_36m_cohort": COHORT_PATH,
}

# Treat null, whitespace-only strings, and explicit missing tokens as missing.
# Count -1 separately because it is not a universal missing-value marker.
MISSING_TOKENS = ["nan", "null", "none", "n/a", "na"]
COLUMNS_PER_BATCH = 20  # Limits memory use on the large raw Parquet file.
missingness_rows = []

for dataset_name, dataset_path in missingness_datasets.items():
    lazy = pl.scan_parquet(dataset_path)
    schema = lazy.collect_schema()
    column_names = schema.names()
    total_rows = pq.ParquetFile(dataset_path).metadata.num_rows

    print(f"Auditing missingness: {dataset_name} ({len(column_names)} columns)")

    for start in range(0, len(column_names), COLUMNS_PER_BATCH):
        batch = column_names[start:start + COLUMNS_PER_BATCH]
        expressions = []

        for col_name in batch:
            value = pl.col(col_name)
            text_value = value.cast(pl.String, strict=False).str.strip_chars()
            lowered = text_value.str.to_lowercase()

            expressions.extend([
                value.is_null().sum().alias(f"{col_name}__null"),
                (text_value == "").fill_null(False).sum().alias(
                    f"{col_name}__blank"
                ),
                lowered.is_in(MISSING_TOKENS).fill_null(False).sum().alias(
                    f"{col_name}__token"
                ),
                (text_value == "-1").fill_null(False).sum().alias(
                    f"{col_name}__minus_one"
                ),
            ])

        summary = lazy.select(expressions).collect().to_dicts()[0]

        for col_name in batch:
            null_count = int(summary[f"{col_name}__null"])
            blank_count = int(summary[f"{col_name}__blank"])
            token_count = int(summary[f"{col_name}__token"])
            minus_one_count = int(summary[f"{col_name}__minus_one"])
            missing_count = null_count + blank_count + token_count

            missingness_rows.append({
                "dataset": dataset_name,
                "column": col_name,
                "dtype": str(schema[col_name]),
                "total_rows": total_rows,
                "null_count": null_count,
                "blank_count": blank_count,
                "missing_text_count": token_count,
                "missing_count": missing_count,
                "missing_pct": round(
                    100 * missing_count / total_rows, 4
                ) if total_rows else 0.0,
                "nonmissing_count": total_rows - missing_count,
                "minus_one_count": minus_one_count,
                "missing_or_minus_one_pct": round(
                    100 * (missing_count + minus_one_count) / total_rows, 4
                ) if total_rows else 0.0,
            })

missingness_report = pd.DataFrame(missingness_rows)
missingness_report = missingness_report.sort_values(
    ["dataset", "missing_pct", "column"],
    ascending=[True, False, True],
).reset_index(drop=True)

missingness_path = AUDIT_REPORT_DIR / "09_full_column_missingness.csv"
missingness_report.to_csv(missingness_path, index=False)

print("Saved audit report:", missingness_path)
print("Rows in report:", len(missingness_report))
display(missingness_report.groupby("dataset", as_index=False).agg(
    columns_audited=("column", "size"),
    columns_with_missing=("missing_count", lambda x: int((x > 0).sum())),
    columns_with_minus_one=("minus_one_count", lambda x: int((x > 0).sum())),
))


Auditing missingness: full_source (142 columns)
Auditing missingness: resolved_2013_2016_36m_cohort (146 columns)
Saved audit report: d:\ML_Projects\P2P-Lending-Default-Prediction\reports\data_audit\09_full_column_missingness.csv
Rows in report: 288


,dataset,columns_audited,columns_with_missing,columns_with_minus_one
0,full_source,142,140,1
1,resolved_2013_2016_36m_cohort,146,67,1


## 00.9 — Duplicate loan IDs

Check missing IDs and duplicate excess rows in the full source and cohort using only the ID column. Export one CSV; no loan IDs are saved.

In [10]:
# 10 — Duplicate ID audit (source + selected cohort)
# Missing/blank IDs are separate from repeated valid IDs.
import pandas as pd
import polars as pl
import pyarrow.parquet as pq
from IPython.display import display

AUDIT_REPORT_DIR = ROOT / "reports" / "data_audit"
AUDIT_REPORT_DIR.mkdir(parents=True, exist_ok=True)

id_datasets = {
    "full_source": RAW_PARQUET,
    "resolved_2013_2016_36m_cohort": COHORT_PATH,
}

id_audit_rows = []

for dataset_name, dataset_path in id_datasets.items():
    lazy = pl.scan_parquet(dataset_path)
    schema = lazy.collect_schema()
    total_rows = pq.ParquetFile(dataset_path).metadata.num_rows

    if "id" not in schema.names():
        id_audit_rows.append({
            "dataset": dataset_name,
            "total_rows": total_rows,
            "blank_or_null_id_rows": None,
            "valid_id_rows": None,
            "unique_valid_ids": None,
            "duplicate_excess_rows": None,
            "duplicate_excess_pct": None,
            "status": "ID_COLUMN_ABSENT",
        })
        continue

    # Select only the ID column to avoid materializing full loan records.
    valid_ids = (
        lazy.select(
            pl.col("id").cast(pl.String, strict=False)
            .str.strip_chars().alias("_audit_id")
        )
        .filter(
            pl.col("_audit_id").is_not_null()
            & (pl.col("_audit_id") != "")
        )
    )

    counts = valid_ids.select(
        pl.len().alias("valid_id_rows"),
        pl.col("_audit_id").n_unique().alias("unique_valid_ids"),
    ).collect().to_dicts()[0]

    valid_count = int(counts["valid_id_rows"])
    unique_count = int(counts["unique_valid_ids"])
    blank_count = total_rows - valid_count
    duplicate_excess = valid_count - unique_count

    id_audit_rows.append({
        "dataset": dataset_name,
        "total_rows": total_rows,
        "blank_or_null_id_rows": blank_count,
        "valid_id_rows": valid_count,
        "unique_valid_ids": unique_count,
        "duplicate_excess_rows": duplicate_excess,
        "duplicate_excess_pct": round(
            100 * duplicate_excess / valid_count, 4
        ) if valid_count else 0.0,
        "status": "PASS" if not (blank_count or duplicate_excess) else "REVIEW",
    })

id_audit_report = pd.DataFrame(id_audit_rows)
id_report_path = AUDIT_REPORT_DIR / "10_duplicate_id_audit.csv"
id_audit_report.to_csv(id_report_path, index=False)

print("Saved audit report:", id_report_path)
display(id_audit_report)


Saved audit report: d:\ML_Projects\P2P-Lending-Default-Prediction\reports\data_audit\10_duplicate_id_audit.csv


,dataset,total_rows,blank_or_null_id_rows,valid_id_rows,unique_valid_ids,duplicate_excess_rows,duplicate_excess_pct,status
0,full_source,2925493,0,2925493,2925493,0,0.0,PASS
1,resolved_2013_2016_36m_cohort,869540,0,869540,869540,0,0.0,PASS


## 00.10 — Dataset inventory

Compare available Parquet artifacts and export file sizes, row counts, and column counts.

In [11]:

from pathlib import Path
import pyarrow.parquet as pq
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

paths = {
    "Full ZIP conversion": ROOT / "data/staging/lendingclub_full_raw.parquet",
    "Interim raw": ROOT / "data/interim/lendingclub_2013_2016_36m_raw.parquet",
    "Interim cohort": ROOT / "data/interim/lendingclub_2013_2016_36m.parquet",
}

records = []

for name, path in paths.items():

    if not path.exists():
        print(f"NOT FOUND: {path}")
        continue

    pf = pq.ParquetFile(path)
    schema = pf.schema_arrow

    row = {
        "dataset": name,
        "rows": pf.metadata.num_rows,
        "columns": len(schema.names),
    }

    for col in [
        "annual_inc",
        "open_acc_6m",
        "issue_d",
        "issue_date",
        "issue_year",
        "term_months",
        "target",
    ]:
        row[col] = (
            str(schema.field(col).type)
            if col in schema.names
            else "ABSENT"
        )

    records.append(row)

display(pd.DataFrame(records).set_index("dataset"))


# CSV REPORT: raw archive and dataset artifact inventory (metadata only)
AUDIT_REPORT_DIR = ROOT / "reports" / "data_audit"
AUDIT_REPORT_DIR.mkdir(parents=True, exist_ok=True)

inventory_items = [
    ("Source ZIP archive", ROOT / "data/raw/archive.zip", "ZIP")
] + [
    (name, path, "Parquet") for name, path in paths.items()
]

inventory_rows = []

for name, path, file_type in inventory_items:
    exists = path.exists()

    row = {
        "dataset": name,
        "file_type": file_type,
        "path": str(path),
        "exists": exists,
        "size_mb": round(path.stat().st_size / (1024 ** 2), 2)
                   if exists else None,
        "rows": None,
        "columns": None,
    }

    if exists and file_type == "Parquet":
        file_metadata = pq.ParquetFile(path).metadata
        row["rows"] = file_metadata.num_rows
        row["columns"] = file_metadata.num_columns

    inventory_rows.append(row)

inventory_report = pd.DataFrame(inventory_rows)

inventory_report_path = AUDIT_REPORT_DIR / "01_dataset_inventory.csv"
inventory_report.to_csv(inventory_report_path, index=False)

print("Saved audit report:", inventory_report_path)


,rows,columns,annual_inc,open_acc_6m,issue_d,issue_date,issue_year,term_months,target
dataset,,,,,,,,,
Full ZIP conversion,2925493,142,string,string,string,ABSENT,ABSENT,ABSENT,ABSENT
Interim raw,869540,146,large_string,large_string,large_string,date32[day],int32,int16,int8
Interim cohort,869540,142,float,int16,timestamp[ns],ABSENT,ABSENT,ABSENT,int8


Saved audit report: d:\ML_Projects\P2P-Lending-Default-Prediction\reports\data_audit\01_dataset_inventory.csv
